In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser
from typing import List

# Carregar variáveis de ambiente
load_dotenv()
chat = ChatOpenAI(api_key = os.getenv("OPENAI_API_KEY"),
                  model="gpt-5-mini", temperature=0.7)

In [3]:
curriculo = """
Nome: João da Silva
E-mail: joao.silva@email.com
Telefone: (11) 98765-4321

Educação:
- Graduação em Ciência da Computação, Universidade ABC, 2015-2019
- Mestrado em Inteligência Artificial, Universidade XYZ, 2020-2022

Experiência Profissional:
- Desenvolvedor de Software Sênior, Tech Solutions Inc., 2022-Presente
  - Responsável por liderar a equipe de desenvolvimento.
- Analista de Dados, Data Innovators, 2019-2022
  - Análise de grandes volumes de dados para otimização de processos.
"""

### 1. Definir o esquema de saída com Pydantic

In [4]:
class Experiencia(BaseModel):
    """Um item de experiência profissional."""
    cargo: str = Field(..., description="O cargo ocupado.")
    empresa: str = Field(..., description="O nome da empresa.")
    periodo: str = Field(..., description="O período em que trabalhou na empresa.")

class Curriculo(BaseModel):
    """Informações extraídas de um currículo."""
    nome: str = Field(..., description="Nome completo da pessoa.")
    email: str = Field(..., description="Endereço de e-mail.")
    telefone: str = Field(..., description="Número de telefone.")
    experiencia: List[Experiencia] = Field(..., description="Lista da experiência profissional da pessoa.")
    
parser_curriculo = JsonOutputParser(pydantic_object=Curriculo)


### 2. Apresentando o Prompt com instruções do parser

In [6]:
prompt_textual = f"""
Sua tarefa é extrair informações chave de um currículo e formatá-las como um objeto JSON.

As instruções de formatação JSON estão abaixo. Certifique-se de seguir o formato e os tipos de dados exatamente como especificado.

{parser_curriculo.get_format_instructions()}

Texto do Currículo:
{curriculo}
"""

### 3. Invocar o LLM e analisar a resposta

In [12]:
resposta_curriculo = chat.invoke(prompt_textual)
resultado_curriculo = parser_curriculo.parse(resposta_curriculo.content)

### 4. Usar o Parser para analisar a resposta

In [13]:
print("--- Análise do Currículo ---")
# Use a notação de colchetes para acessar os valores do dicionário
print(f"Nome: {resultado_curriculo['nome']}")
print(f"E-mail: {resultado_curriculo['email']}")
print(f"Telefone: {resultado_curriculo['telefone']}")

print("\nExperiência Profissional:")
for exp in resultado_curriculo['experiencia']:
    print(f"- {exp['cargo']} na {exp['empresa']} ({exp['periodo']})")

--- Análise do Currículo ---
Nome: João da Silva
E-mail: joao.silva@email.com
Telefone: (11) 98765-4321

Experiência Profissional:
- Desenvolvedor de Software Sênior na Tech Solutions Inc. (2022-Presente)
- Analista de Dados na Data Innovators (2019-2022)
